# Multiclass Attack Classification

This notebook predicts the attack category: `normal`, `dos`, `probe`, `r2l`, or `u2r`. This is harder than binary detection because minority attack categories are rare.


In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)


/Users/paco/Documents/Git Projects/network-intrusion-detection-system


In [2]:
import json

import pandas as pd

from src.config import CLEANED_DIR, FEATURE_COLUMNS, MODELS_DIR, REPORTS_DIR, FIGURES_DIR, ensure_project_dirs
from src.modeling import evaluate_multiclass, metrics_to_frame, multiclass_models, save_model
from src.visualize import save_metric_barplot

ensure_project_dirs()
train_df = pd.read_csv(CLEANED_DIR / "train_cleaned.csv")
test_df = pd.read_csv(CLEANED_DIR / "test_cleaned.csv")

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df["attack_category"]
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df["attack_category"]


## Train and Evaluate Multiclass Models


In [3]:
models = multiclass_models()
results = {}
fitted_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    fitted_models[name] = model
    results[name] = evaluate_multiclass(model, X_test, y_test)

multiclass_results = metrics_to_frame(results).sort_values("weighted_f1", ascending=False)
display(multiclass_results)
multiclass_results.to_csv(REPORTS_DIR / "multiclass_model_comparison.csv", index=False)
save_metric_barplot(multiclass_results, "weighted_f1", FIGURES_DIR / "multiclass_weighted_f1_comparison.png", "Multiclass Attack Classification - Weighted F1")


Training Random Forest...
Training HistGradientBoosting...


,model,accuracy,macro_f1,weighted_f1
1,HistGradientBoosting,0.781405,0.559780,0.752904
0,Random Forest,0.734430,0.475069,0.686953


## Save Best Multiclass Model


In [4]:
best_multiclass_name = multiclass_results.iloc[0]["model"]
best_multiclass_model = fitted_models[best_multiclass_name]
save_model(best_multiclass_model, MODELS_DIR / "multiclass_attack_model.pkl")

with open(REPORTS_DIR / "multiclass_metrics.json", "w", encoding="utf-8") as file:
    json.dump(results, file, indent=2)

print("Best multiclass model:", best_multiclass_name)


Best multiclass model: HistGradientBoosting


## Per-Class Notes


In [5]:
best_report = results[best_multiclass_name]["classification_report"]
pd.DataFrame(best_report).T


,precision,recall,f1-score,support
dos,0.964872,0.835791,0.895705,7460.000000
normal,0.698842,0.969210,0.812114,9711.000000
probe,0.768500,0.634862,0.695318,2421.000000
r2l,0.807692,0.145581,0.246696,2885.000000
u2r,0.127660,0.179104,0.149068,67.000000
accuracy,0.781405,0.781405,0.781405,0.781405
macro avg,0.673513,0.552910,0.559780,22544.000000
weighted avg,0.806586,0.781405,0.752904,22544.000000
